# Train and register the inference model
Run all cells once. The dataset stays in memory. The final cell pins the registered version for both deployments. Re-running creates a new model version.


In [1]:
import os
from pathlib import Path
import boto3
import mlflow
import mlflow.sklearn
import pandas as pd
from botocore.exceptions import ClientError
from mlflow.tracking import MlflowClient
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from ucimlrepo import fetch_ucirepo

tracking_uri = os.environ["MLFLOW_TRACKING_URI"]
mlflow.set_tracking_uri(tracking_uri)
mlflow.set_registry_uri(tracking_uri)
bucket = os.environ["S3_BUCKET"]
s3 = boto3.client("s3", endpoint_url=os.environ["MLFLOW_S3_ENDPOINT_URL"])
try:
    s3.head_bucket(Bucket=bucket)
except ClientError as exc:
    if str(exc.response["Error"]["Code"]) not in {"404", "NoSuchBucket", "NotFound"}:
        raise
    s3.create_bucket(Bucket=bucket)
print(f"Artifact bucket ready: {bucket}")

Artifact bucket ready: mlflow-artifacts


In [2]:
dataset = fetch_ucirepo(id=45)
features = dataset.data.features.apply(pd.to_numeric, errors="coerce")
target = pd.to_numeric(dataset.data.targets.iloc[:, 0], errors="coerce")
valid = features.notna().all(axis=1) & target.notna()
X = features.loc[valid].astype("float64")
y = (target.loc[valid] > 0).astype("int64")
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Training rows: {len(X_train)}, test rows: {len(X_test)}")
X.head()

Training rows: 237, test rows: 60


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal
0,63.0,1.0,1.0,145.0,233.0,1.0,2.0,150.0,0.0,2.3,3.0,0.0,6.0
1,67.0,1.0,4.0,160.0,286.0,0.0,2.0,108.0,1.0,1.5,2.0,3.0,3.0
2,67.0,1.0,4.0,120.0,229.0,0.0,2.0,129.0,1.0,2.6,2.0,2.0,7.0
3,37.0,1.0,3.0,130.0,250.0,0.0,0.0,187.0,0.0,3.5,3.0,0.0,3.0
4,41.0,0.0,2.0,130.0,204.0,0.0,2.0,172.0,0.0,1.4,1.0,0.0,3.0


In [3]:
mlflow.set_experiment("assignment5-heart-disease")
model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=1)
with mlflow.start_run(run_name="initial-model") as run:
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    mlflow.log_params({"n_estimators": 100, "random_state": 42, "n_jobs": 1, "dataset": "UCI Heart Disease 45"})
    mlflow.log_metrics({"accuracy": accuracy_score(y_test, predictions), "f1": f1_score(y_test, predictions)})
    info = mlflow.sklearn.log_model(model, name="model", input_example=X_train.head(3))
    run_id = run.info.run_id

# Register the artifact location directly, as in assignment4, for MLflow 3.5 compatibility.
client = MlflowClient()
model_name = "HeartDiseaseClassifier"
if not client.search_registered_models(f"name='{model_name}'"):
    client.create_registered_model(model_name)
logged_model = client.get_logged_model(info.model_uri.removeprefix("models:/"))
version = client.create_model_version(
    name=model_name, source=logged_model.artifact_location,
    run_id=run_id, await_creation_for=60,
)
client.set_registered_model_alias(model_name, "champion", str(version.version))
model_uri = f"models:/{model_name}/{version.version}"
loaded = mlflow.sklearn.load_model(model_uri)
assert (loaded.predict(X_test) == model.predict(X_test)).all()

output = Path("/workspace/output/model.env")
output.write_text(f"MLFLOW_MODEL_URI={model_uri}\n")
print(f"Registered champion: {model_uri}")
print(f"Deployment version saved to {output}")

2026/10/14 04:23:01 INFO mlflow.tracking.fluent: Experiment with name 'assignment5-heart-disease' does not exist. Creating a new experiment.
2026/10/14 04:23:02 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably not on your PATH), so Git SHA is not available. Error: Failed to initialize: Bad git executable.
The git executable must be specified in one of the following ways:
    - be included in your $PATH
    - be set via $GIT_PYTHON_GIT_EXECUTABLE
    - explicitly set via git.refresh(<full-path-to-git-executable>)

All git commands will error until this is rectified.

This initial message can be silenced or aggravated in the future by setting the
$GIT_PYTHON_REFRESH environment variable. Use one of the following values:
    - quiet|q|silence|s|silent|none|n|0: for no message or exception
    - warn|w|warning|log|l|1: for a warning message (logging level CRITICAL, displayed by default)
    - error|e|exception|raise|r|2: for a raised exception

Example:


🏃 View run initial-model at: http://mlflow:5000/#/experiments/1/runs/55eb8b9bf847472a820b5a3ac4acc0a6
🧪 View experiment at: http://mlflow:5000/#/experiments/1


2026/10/14 04:23:08 INFO mlflow.store.model_registry.abstract_store: Waiting up to 60 seconds for model version to finish creation. Model name: HeartDiseaseClassifier, version 1


Registered champion: models:/HeartDiseaseClassifier/1
Deployment version saved to /workspace/output/model.env
